# 7. Otimizacao (calibracao WOFOST) -- Soja no Parana

Etapa final: calibra os parametros do WOFOST por cluster para que a produtividade
simulada reproduza o mais proximo possivel a produtividade destendenciada do IBGE
(`dyield`, etapa 5), usando a estrategia **multi-anual** (todas as safras 2015/16-2024/25
como alvos simultaneos), que e a recomendada em `util/NLOPT_MultiYear.py` por reduzir
drasticamente o tempo de calibracao frente a otimizar ano a ano.

Usa `SoyWOFOSTMultiYearOptimizerPR` (`util/NLOPT_soja_pr.py`), que:
- reaproveita toda a mecanica de otimizacao (NLOPT, funcao objetivo RMSE multi-anual,
  configuracao das tabelas de parametros do WOFOST, calculo de metricas, persistencia de
  resultados) de `WOFOSTMultiYearOptimizer` sem modificar `util/NLOPT.py` /
  `util/NLOPT_MultiYear.py`;
- troca a cultura ativa para `soybean` / `Soybean_VanHeemst_1988` e o calendario para o da
  soja no PR;
- carrega o `CLUSTER_PARAMS` (quais parametros calibrar, em ordem de importancia, por
  cluster) a partir do ranking gerado na etapa 6, em vez do dicionario fixo calibrado
  para milho.


In [1]:
import json
import os
import sys
import time

import nlopt

sys.path.append(os.path.join(os.getcwd(), 'util'))
from util.SensitivityAnalyzer_soja_pr import SoyNetCDFDataLoader
from util.NLOPT_soja_pr import SoyWOFOSTMultiYearOptimizerPR, load_cluster_params_from_ranking
from util.utils_soja_pr import setup_paths_soja_pr

paths = setup_paths_soja_pr()

ranking_json_path = os.path.join(paths['RESULTS'], 'SA_ranking_by_cluster.json')
cluster_params = load_cluster_params_from_ranking(ranking_json_path, top_n=44)

print("Clusters com ranking de sensibilidade disponivel:", list(cluster_params.keys()))


Clusters com ranking de sensibilidade disponivel: [2.0, 1.0, 0.0, 3.0]


In [2]:
nc_loader = SoyNetCDFDataLoader(paths['COMPLETO'])

optimizer = SoyWOFOSTMultiYearOptimizerPR(
    paths=paths,
    nc_loader=nc_loader,
    cluster_params=cluster_params,
    algorithm=nlopt.LN_BOBYQA,
    max_eval=3000,
)


Encontrados 392 arquivos NetCDF


## Rodar a calibracao para todos os municipios, agrupados por cluster


In [3]:
start_time = time.time()

all_files = nc_loader.nc_files
files_by_cluster = {}
for nc_file in all_files:
    point_info = nc_loader.get_point_info(nc_file)
    cluster_id = point_info['cluster_id']
    files_by_cluster.setdefault(cluster_id, []).append(nc_file)

all_results = []
for cluster_id in sorted(files_by_cluster.keys()):
    files = files_by_cluster[cluster_id]

    if cluster_id not in cluster_params:
        print(f"[pulado] Cluster {cluster_id}: sem ranking de sensibilidade (rode a etapa 6 antes).")
        continue

    print(f"\n{'=' * 60}\nCLUSTER {cluster_id}: {len(files)} municipios\n{'=' * 60}")

    for i, nc_file in enumerate(files, 1):
        print(f"\n[{i}/{len(files)}]")
        result = optimizer.optimize_point_multiyear(nc_file, cluster_id)
        if result is not None:
            all_results.append({
                'point_id': result['point_id'],
                'cluster_id': result['cluster_id'],
                'rmse_aggregated': result['rmse_aggregated'],
                'n_years': result['n_years'],
            })

elapsed = time.time() - start_time
print(f"\nOtimizacao concluida em {elapsed / 3600:.2f} horas. {len(all_results)} municipios calibrados.")



CLUSTER 0.0: 191 municipios

[1/191]

🎯 OTIMIZAÇÃO MULTI-ANUAL - Point 4100103 (Cluster 0.0)
📦 Preparando dados de todos os anos...


[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'


📅 Anos com dados válidos: 10
   Anos: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

🔧 Iniciando otimização hierárquica (9 etapas)

──────────────────────────────────────────────────────────────────────
📊 Etapa 1/9: Otimizando 5 parâmetros
   Novos parâmetros: ['DVSEND', 'TMPFTB035', 'AMAXTB082', 'AMAXTB200', 'TMPFTB020']


[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'


   ✅ RMSE: 3.57 kg/ha (52.6s)

🎉 RMSE < 100 kg/ha alcançado! Interrompendo otimização.

📈 Calculando métricas por ano...
   💾 Parâmetros salvos: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100103_params.csv
   💾 Métricas anuais salvas: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100103_yearly_metrics.csv
   💾 Resumo salvo: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100103_summary.json

🎉 Otimização PERFEITO ✨!
   RMSE agregado: 3.57 kg/ha
   Tempo total: 59.7s
   Anos processados: 10


[2/191]

🎯 OTIMIZAÇÃO MULTI-ANUAL - Point 4100608 (Cluster 0.0)
📦 Preparando dados de todos os anos...
📅 Anos com dados válidos: 10
   Anos: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

🔧 Iniciando otimização hierárquica (9 etapas)

────────────────────────────────────────────────────────

...

   ✅ RMSE: 2.72 kg/ha (53.4s)

🎉 RMSE < 100 kg/ha alcançado! Interrompendo otimização.

📈 Calculando métricas por ano...
   💾 Parâmetros salvos: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100806_params.csv
   💾 Métricas anuais salvas: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100806_yearly_metrics.csv
   💾 Resumo salvo: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100806_summary.json

🎉 Otimização PERFEITO ✨!
   RMSE agregado: 2.72 kg/ha
   Tempo total: 57.3s
   Anos processados: 10


[4/191]

🎯 OTIMIZAÇÃO MULTI-ANUAL - Point 4100905 (Cluster 0.0)
📦 Preparando dados de todos os anos...
📅 Anos com dados válidos: 10
   Anos: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

🔧 Iniciando otimização hierárquica (9 etapas)

────────────────────────────────────────────────────────

[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'


   ✅ RMSE: 2.19 kg/ha (57.4s)

🎉 RMSE < 100 kg/ha alcançado! Interrompendo otimização.

📈 Calculando métricas por ano...
   💾 Parâmetros salvos: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100905_params.csv
   💾 Métricas anuais salvas: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100905_yearly_metrics.csv
   💾 Resumo salvo: d:\_py\AgroIA_prod\output\soja_pr\Optimization\OPT_MULTIYEAR_cluster0.0_point4100905_summary.json

🎉 Otimização PERFEITO ✨!
   RMSE agregado: 2.19 kg/ha
   Tempo total: 62.4s
   Anos processados: 10


[5/191]

🎯 OTIMIZAÇÃO MULTI-ANUAL - Point 4101101 (Cluster 0.0)
📦 Preparando dados de todos os anos...
📅 Anos com dados válidos: 10
   Anos: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

🔧 Iniciando otimização hierárquica (9 etapas)

────────────────────────────────────────────────────────

[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'
[ERROR] - Erro na simulação: 'NoneType' object has no attribute 'add_variable'


KeyboardInterrupt: 

In [ ]:
summary_file = os.path.join(paths['OPTIMIZATION'], 'optimization_summary_pr.json')
with open(summary_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Resumo salvo em {summary_file}")
print(f"Resultados detalhados por municipio (parametros calibrados + metricas anuais) em: {paths['OPTIMIZATION']}")
